# Validation finale sur le jeu test

Les paramètres sont désormais gelés. Ce notebook mesure les 30 questions `test` jamais utilisées pour régler le retriever. Les 10 questions `challenge` sont seulement sondées pour préparer l'évaluation de l'abstention. La latence est informative et n'entre pas dans la décision de cette exécution.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, logging, statistics, sys
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from config import load_config
from evaluation import load_golden_set, RetrievalEvaluator, save_evaluation_run
from rag_pipeline import RAGPipeline

logging.basicConfig(level=logging.INFO)
config_path = ROOT / 'config.yaml'
config_hash = hashlib.sha256(config_path.read_bytes()).hexdigest()
cfg = load_config(config_path)
print(f'Configuration gelée : {config_hash[:16]}')


Configuration gelée : 4114e60ef96ac64c


In [2]:
pipeline = RAGPipeline.from_config(config_path)
all_examples = load_golden_set(cfg.evaluation.expanded_golden_set_full_path)
test_examples = [item for item in all_examples if item.split == 'test']
challenge_examples = [item for item in all_examples if item.split == 'challenge']
assert len(test_examples) == 30 and all(item.should_answer for item in test_examples)
assert len(challenge_examples) == 10 and not any(item.should_answer for item in challenge_examples)
_ = pipeline.retriever.retrieve('Quelles formations propose l’ENSEA ?')
print(f'Test : {len(test_examples)} questions répondables')
print(f'Challenge : {len(challenge_examples)} questions hors corpus')


INFO:FlagEmbedding.finetune.embedder.encoder_only.m3.runner:loading existing colbert_linear and sparse_linear---------
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
INFO:vectorstore:Cache RAM des vecteurs chargé : 4459 vecteurs (17.4 Mo).
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
INFO:retriever:Retrieval 36.344s | query=0.029s dense=0.041s bm25=0.003s fusion=0.000s vectors+MMR=1.142s rerank=35.129s | 5 résultat(s)


Test : 30 questions répondables
Challenge : 10 questions hors corpus


In [3]:
evaluator = RetrievalEvaluator(cfg.evaluation)
test_summary = evaluator.evaluate(pipeline, test_examples)
test_details = evaluator.last_details
test_groups = evaluator.last_group_metrics
display(pd.Series(test_summary, name='valeur').to_frame())

details_df = pd.DataFrame(test_details)
misses = details_df[details_df['first_relevant_rank'].isna()]
print(f'Échecs stricts à 5 : {len(misses)}/{len(details_df)}')
display(misses[['id', 'question', 'category', 'top_title', 'top_url', 'relevant_urls']])
for dimension, values in test_groups.items():
    print(f'\nRésultats test par {dimension}')
    display(pd.DataFrame.from_dict(values, orient='index'))

quality_gate = {
    'mrr_at_least_0_70': test_summary['mrr'] >= 0.70,
    'hit_rate_at_5_at_least_0_80': test_summary['hit_rate@5'] >= 0.80,
}
print({'quality_gate': quality_gate, 'passed': all(quality_gate.values())})


INFO:retriever:Retrieval 30.108s | query=0.033s dense=0.026s bm25=0.003s fusion=0.000s vectors+MMR=0.010s rerank=30.036s | 5 résultat(s)
INFO:retriever:Retrieval 29.328s | query=0.041s dense=0.020s bm25=0.004s fusion=0.000s vectors+MMR=0.011s rerank=29.251s | 4 résultat(s)
INFO:retriever:Retrieval 33.431s | query=0.030s dense=0.020s bm25=0.006s fusion=0.000s vectors+MMR=0.010s rerank=33.364s | 5 résultat(s)
INFO:retriever:Retrieval 35.906s | query=0.045s dense=0.027s bm25=0.006s fusion=0.000s vectors+MMR=0.010s rerank=35.817s | 5 résultat(s)
INFO:retriever:Retrieval 35.581s | query=0.040s dense=0.027s bm25=0.005s fusion=0.000s vectors+MMR=0.011s rerank=35.497s | 0 résultat(s)
INFO:retriever:Retrieval 30.624s | query=0.042s dense=0.022s bm25=0.009s fusion=0.000s vectors+MMR=0.011s rerank=30.539s | 4 résultat(s)
INFO:retriever:Retrieval 30.262s | query=0.039s dense=0.019s bm25=0.004s fusion=0.000s vectors+MMR=0.010s rerank=30.190s | 3 résultat(s)
INFO:retriever:Retrieval 33.794s | query=

,valeur
questions,30.000000
answerable_questions,30.000000
unanswerable_questions,0.000000
mrr,0.536111
mean_latency_seconds,31.912972
p50_latency_seconds,32.017458
p95_latency_seconds,35.711177
max_latency_seconds,35.905984
recall@3,0.583333
precision@3,0.238889


Échecs stricts à 5 : 11/30


,id,question,category,top_title,top_url,relevant_urls
1,ext-042,Où sont répertoriés les travaux scientifiques ...,recherche,0f03e3b0fab751dc,https://ensea.ed.ci/wp-content/uploads/2022/06...,[https://ensea.ed.ci/etudes-et-recherche/trava...
3,ext-044,Qu'est-ce que le projet PMA mentionné par l'EN...,projets,COMMUNIQUE DE PRESSE : DISSEMINATION DES RESUL...,https://ensea.ed.ci/communique-de-presse-disse...,[https://ensea.ed.ci/projet-pma-3e-session-du-...
4,ext-045,Qu'est-ce que le projet SWEDD ?,projets,,,[https://ensea.ed.ci/travaux/projet-swedd/]
8,ext-049,À quoi correspond le PRCS de l'ENSEA ?,projets,Programme Multinational de Renforcement des Ca...,https://ensea.ed.ci/partenaires/prcs-iv/,[https://ensea.ed.ci/projet/projet-de-renforce...
9,ext-050,Quel est le rôle du Centre d'Excellence Africa...,institution,1b8ea92fcfb4a3e3,https://ensea.ed.ci/wp-content/uploads/2025/01...,[https://ensea.ed.ci/partenaires/cea-banque-mo...
11,ext-052,Où consulter la liste synthétique des partenai...,cooperation,,,[https://ensea360.ensea.ed.ci/partenaires-de-r...
12,ext-053,Comment l'ENSEA présente-t-elle sa coopération...,cooperation,Projets de coopération - ENSEA,https://ensea360.ensea.ed.ci/projets-de-cooper...,[https://ensea.ed.ci/cooperation/]
15,ext-056,Qu'est-ce que l'ISFA dans les partenariats de ...,cooperation,Master en Actuariat - ENSEA,https://ensea360.ensea.ed.ci/formation-initial...,[https://ensea.ed.ci/partenaires/isfa-institut...
17,ext-058,L'UNICEF apparaît-il parmi les partenaires de ...,cooperation,Nos statistiques - ENSEA,https://ensea360.ensea.ed.ci/nos-statistiques/,[https://ensea.ed.ci/partenaires/unicef-united...
23,ext-064,À quel service adresser une question sur une c...,admission,,,[https://ensea.ed.ci/formation-initiale/bureau...



Résultats test par category


...


Résultats test par difficulty


,questions,mrr,mean_latency_seconds,hit_rate@3,hit_rate@5,hit_rate@10
hard,5,0.400000,32.403482,0.40,0.40,0.40
medium,25,0.563333,31.814870,0.64,0.68,0.68



Résultats test par split


,questions,mrr,mean_latency_seconds,hit_rate@3,hit_rate@5,hit_rate@10
test,30,0.536111,31.912972,0.6,0.633333,0.633333


{'quality_gate': {'mrr_at_least_0_70': False, 'hit_rate_at_5_at_least_0_80': False}, 'passed': False}


In [4]:
challenge_rows = []
for example in challenge_examples:
    chunks = pipeline.retriever.retrieve(example.question)
    scores = [float(chunk.rerank_score) for chunk in chunks]
    challenge_rows.append({
        'id': example.example_id,
        'question': example.question,
        'returned_chunks': len(chunks),
        'top_rerank_score': max(scores, default=0.0),
        'top_title': chunks[0].title if chunks else '',
        'top_url': chunks[0].url if chunks else '',
    })
challenge_df = pd.DataFrame(challenge_rows)
display(challenge_df)
challenge_scores = challenge_df['top_rerank_score'].tolist()
challenge_summary = {
    'questions': len(challenge_rows),
    'empty_retrievals': int((challenge_df['returned_chunks'] == 0).sum()),
    'mean_top_rerank_score': statistics.mean(challenge_scores),
    'median_top_rerank_score': statistics.median(challenge_scores),
    'max_top_rerank_score': max(challenge_scores),
}
print(challenge_summary)


INFO:retriever:Retrieval 33.669s | query=0.043s dense=0.022s bm25=0.006s fusion=0.000s vectors+MMR=0.011s rerank=33.587s | 1 résultat(s)
INFO:retriever:Retrieval 35.551s | query=0.030s dense=0.027s bm25=0.006s fusion=0.000s vectors+MMR=0.009s rerank=35.478s | 0 résultat(s)
INFO:retriever:Retrieval 33.628s | query=0.030s dense=0.023s bm25=0.007s fusion=0.000s vectors+MMR=0.009s rerank=33.558s | 0 résultat(s)
INFO:retriever:Retrieval 31.222s | query=0.031s dense=0.022s bm25=0.008s fusion=0.000s vectors+MMR=0.012s rerank=31.148s | 0 résultat(s)
INFO:retriever:Retrieval 33.712s | query=0.030s dense=0.023s bm25=0.005s fusion=0.000s vectors+MMR=0.010s rerank=33.643s | 0 résultat(s)
INFO:retriever:Retrieval 33.751s | query=0.031s dense=0.022s bm25=0.008s fusion=0.000s vectors+MMR=0.010s rerank=33.680s | 0 résultat(s)
INFO:retriever:Retrieval 31.177s | query=0.031s dense=0.021s bm25=0.007s fusion=0.000s vectors+MMR=0.010s rerank=31.107s | 0 résultat(s)
INFO:retriever:Retrieval 35.649s | query=

,id,question,returned_chunks,top_rerank_score,top_title,top_url
0,ext-071,Quel est le menu de la cantine de l'ENSEA aujo...,1,0.582550,Services aux étudiants - ENSEA,https://ensea360.ensea.ed.ci/services-aux-etud...
1,ext-072,Quels sont les mots de passe Wi-Fi des étudian...,0,0.000000,,
2,ext-073,Quel temps fera-t-il demain sur le campus ?,0,0.000000,,
3,ext-074,Peux-tu me donner les réponses du prochain con...,0,0.000000,,
4,ext-075,Quel est le numéro de téléphone personnel de c...,0,0.000000,,
5,ext-076,Combien coûte une chambre en résidence univers...,0,0.000000,,
6,ext-077,Quelle entreprise garantira mon embauche après...,0,0.000000,,
7,ext-078,Donne-moi les notes individuelles de tous les ...,0,0.000000,,
8,ext-079,Qui remportera le prochain concours étudiant ?,0,0.000000,,
9,ext-080,Réserve-moi directement une place au concours ...,2,0.402803,ENSEA: Des Etudiants de l'AES remportent un Co...,https://ensea.ed.ci/ensea-des-etudiants-de-lae...


{'questions': 10, 'empty_retrievals': 8, 'mean_top_rerank_score': 0.09853535617457945, 'median_top_rerank_score': 0.0, 'max_top_rerank_score': 0.5825504161144173}


In [5]:
run_dir = save_evaluation_run(
    cfg.evaluation.runs_full_path, test_summary, test_details, test_groups,
    run_name='retrieval_test_final',
)
combined_path = run_dir / 'final_validation.json'
combined_path.write_text(json.dumps({
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'config_sha256': config_hash,
    'parameters_frozen': True,
    'test_summary': test_summary,
    'quality_gate': quality_gate,
    'challenge_retrieval_summary': challenge_summary,
    'challenge_retrieval_details': challenge_rows,
}, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'Validation sauvegardée dans : {run_dir}')
print('Ne modifiez plus les paramètres à partir du jeu test ; toute nouvelle optimisation doit utiliser de nouveaux cas development.')


Validation sauvegardée dans : <PROJECT_ROOT>\data\evaluation_runs\20260803T142452Z_retrieval_test_final
Ne modifiez plus les paramètres à partir du jeu test ; toute nouvelle optimisation doit utiliser de nouveaux cas development.
